# Converting an XML Corpus to a Plaintext Corpus on Disk

This template is like [xml2txt.ipynb](https://github.com/langeslag/ehtc/blob/main/templates/xml2txt.ipynb) except it loads a selection of documents from Corpus Corporum's Patrologia Latina corpus. Because the Corpus Corporum website relies on JavaScript download constructs, actually downloading the XML files is a little involved and relies on an app called Playwright that requires configuring outside of Python.

In [9]:
import re
from pathlib import Path
from lxml import etree
from urllib import request
from playwright.async_api import async_playwright

In [10]:
xml_folder = Path.cwd() / 'xml'
txt_folder = Path.cwd() / 'txt'
for i in [xml_folder, txt_folder]:
    i.mkdir(exist_ok = True)

sources = [
    'https://mlat.uzh.ch/browser/38/1518/3863/8886',
    'https://mlat.uzh.ch/browser/38/1397/3453/8470',
    'https://mlat.uzh.ch/browser/38/1047/3657/8680',
    'https://mlat.uzh.ch/browser/38/1163/21547/21434'
]

In [11]:
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    for page_url in sources:
        page = await browser.new_page(accept_downloads=True)

        await page.goto(page_url)

        async with page.expect_download() as download_info:
            await page.get_by_text(re.compile("Download XML", re.I)).click()

        download = await download_info.value

        filename = str(download.url)[-4:] + '.xml'
        target = xml_folder / filename

        await download.save_as(target)

    await browser.close()

In [12]:
# Discarding unwanted elements:
def simplify(branch):
    discard = ['abbr', 'am', 'sic', 'del', 'note', 'surplus', 'orig', 'fw']  
    # Now we define their text nodes as empty strings:
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        for element in hit.iter():
            element.text = ''
            element.tail = ''
    return branch

In [35]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus = dict()
for file in xml_folder.glob('*.xml'):
    basename = file.stem
    tree = etree.parse(file, parser=parser)
    root = simplify(tree.getroot())
    body = root.find('.//{http://www.tei-c.org/ns/1.0}body')
    segments = dict()
    rubric_counter = 0
    tokens = []
    for identifier in body.iter('{http://www.tei-c.org/ns/1.0}pb'):
        identifier.text = identifier.get('n') + ': '
        if identifier.tail is not None:
            identifier.tail = identifier.tail.lstrip()
    for segment in body.iter('{http://www.tei-c.org/ns/1.0}head', '{http://www.tei-c.org/ns/1.0}p'):
        segment_string = etree.tostring(segment, method='text', encoding='unicode').lstrip()
        if segment.tag == '{http://www.tei-c.org/ns/1.0}p':
            segment_string = segment_string.replace('\n', ' ')
        tokens.append(segment_string)
    corpus[basename] = tokens
        

In [38]:
for ref,doc in corpus.items():
    target_file = txt_folder / str(ref + '.txt')
    with open(target_file, 'w') as f:
        f.write(' '.join(doc))